24.11.2026

## 1.1

Создайте DataFrame, который будет содержать следующие значения:

• id

• name

• age

• salary

• department

• hiredate

• is_active

In [3]:
import pandas as pd
import numpy as np
import os
from time import perf_counter

np.random.seed(1520441)
n = 10000

df = pd.DataFrame({
    "id": np.arange(1, n + 1),
    "name": np.random.choice(["Ivan", "Anna", "Pavel", "Maria", "Olga", "Dmitry"], n),
    "age": np.random.randint(20, 61, n),
    "salary": np.random.randint(40000, 150001, n),
    "department": np.random.choice(["IT", "HR", "Finance"], n),
    "hiredate": pd.Timestamp("2020-01-01") + pd.to_timedelta(np.random.randint(0, 2000, n), unit="D"),
    "is_active": np.random.choice([True, False], n)
})

print(df.head())
print("Размер таблицы:", df.shape)

   id   name  age  salary department   hiredate  is_active
0   1  Pavel   29  109841    Finance 2021-12-14       True
1   2  Pavel   48   68010    Finance 2021-12-31       True
2   3   Anna   42   77485         HR 2021-02-23       True
3   4  Maria   60  127866    Finance 2023-12-30       True
4   5  Maria   55  144693    Finance 2023-01-10       True
Размер таблицы: (10000, 7)


## 2.1

Сохраните DataFrame в CSV двумя способами: обычным и с параметром index=False

In [4]:
df.to_csv("primer_index.csv")
df.to_csv("primer.csv", index=False)
print("Файлы сохранены")

Файлы сохранены


## 2.2

Прочитайте файл обратно и проверьте типы данных

In [5]:
df_csv = pd.read_csv("primer.csv")
print(df_csv.head())
print("Типы до сохранения:")
print(df.dtypes)
print("Типы после чтения CSV:")
print(df_csv.dtypes)

   id   name  age  salary department    hiredate  is_active
0   1  Pavel   29  109841    Finance  2021-12-14       True
1   2  Pavel   48   68010    Finance  2021-12-31       True
2   3   Anna   42   77485         HR  2021-02-23       True
3   4  Maria   60  127866    Finance  2023-12-30       True
4   5  Maria   55  144693    Finance  2023-01-10       True
Типы до сохранения:
id                     int64
name                     str
age                    int32
salary                 int32
department               str
hiredate      datetime64[us]
is_active               bool
dtype: object
Типы после чтения CSV:
id            int64
name            str
age           int64
salary        int64
department      str
hiredate        str
is_active      bool
dtype: object


## 2.3

Сравните размеры файлов (с index и без)

In [7]:
print("С индексом:", os.path.getsize("primer_index.csv"), "байт")
print("Без индекса:", os.path.getsize("primer.csv"), "байт")

С индексом: 470875 байт
Без индекса: 421984 байт


## 2.4

Изучите параметры dtype, parse_dates, usecols при чтении - они ускоряют загрузку

In [10]:
df_csv = pd.read_csv("primer.csv", dtype={"age": "int32"},
                     parse_dates=["hiredate"], usecols=["name", "age", "hiredate"])
print(df_csv.head())
print(df_csv.dtypes)

    name  age   hiredate
0  Pavel   29 2021-12-14
1  Pavel   48 2021-12-31
2   Anna   42 2021-02-23
3  Maria   60 2023-12-30
4  Maria   55 2023-01-10
name                   str
age                  int32
hiredate    datetime64[us]
dtype: object


`dtype` задаёт тип столбца, `parse_dates` преобразует даты, `usecols` позволяет прочитать только нужные столбцы. Влияние на скорость зависит от данных и параметров чтения.

## 3.1

Создайте небольшой DataFrame с вложенным столбцом (например, список навыков сотрудника)

In [11]:
import json

df_json = pd.DataFrame({
    "id": [1, 2, 3],
    "name": ["Ivan", "Anna", "Pavel"],
    "skills": [["Python", "SQL"], ["Excel", "SQL"], ["C++"]]
})
print(df_json)

   id   name         skills
0   1   Ivan  [Python, SQL]
1   2   Anna   [Excel, SQL]
2   3  Pavel          [C++]


## 3.2

Сохраните в JSON в разных ориентациях: records, columns, index, split

In [12]:
orientations = ["records", "columns", "index", "split"]

for orient in orientations:
    df_json.to_json("primer_" + orient + ".json", orient=orient, force_ascii=False)

print("Файлы сохранены")

Файлы сохранены


## 3.3

Сравните размеры и структуру файлов

In [13]:
for orient in orientations:
    filename = "primer_" + orient + ".json"
    print(orient, "-", os.path.getsize(filename), "байт")
    with open(filename, encoding="utf-8") as f:
        print(f.read())
    print()

records - 139 байт
[{"id":1,"name":"Ivan","skills":["Python","SQL"]},{"id":2,"name":"Anna","skills":["Excel","SQL"]},{"id":3,"name":"Pavel","skills":["C++"]}]

columns - 133 байт
{"id":{"0":1,"1":2,"2":3},"name":{"0":"Ivan","1":"Anna","2":"Pavel"},"skills":{"0":["Python","SQL"],"1":["Excel","SQL"],"2":["C++"]}}

index - 151 байт
{"0":{"id":1,"name":"Ivan","skills":["Python","SQL"]},"1":{"id":2,"name":"Anna","skills":["Excel","SQL"]},"2":{"id":3,"name":"Pavel","skills":["C++"]}}

split - 134 байт
{"columns":["id","name","skills"],"index":[0,1,2],"data":[[1,"Ivan",["Python","SQL"]],[2,"Anna",["Excel","SQL"]],[3,"Pavel",["C++"]]]}



records - список записей, columns - данные по столбцам, index - данные по индексам, split - отдельно индексы, названия столбцов и значения.

## 3.4

Прочитайте JSON обратно, примените json_normalize для «разворачивания» вложенности

In [14]:
with open("primer_records.json", encoding="utf-8") as f:
    data = json.load(f)

print(pd.DataFrame(data))

result = pd.json_normalize(data, record_path="skills", meta=["id", "name"])
result = result.rename(columns={0: "skill"})
print(result[["id", "name", "skill"]])

   id   name         skills
0   1   Ivan  [Python, SQL]
1   2   Anna   [Excel, SQL]
2   3  Pavel          [C++]
  id   name   skill
0  1   Ivan  Python
1  1   Ivan     SQL
2  2   Anna   Excel
3  2   Anna     SQL
4  3  Pavel     C++


## 4.1

Сохраните основной DataFrame в Parquet с разными алгоритмами сжатия (snappy, gzip, brotli)

Для работы с Parquet необходимо установить `pyarrow`, а для HDF5 - `tables` (если библиотеки ещё не установлены: `%pip install pyarrow tables`).

При проверке отчёта библиотека pyarrow отсутствовала. Ячейки 4.1–4.5 необходимо выполнить после её установки; результаты Parquet не подставлялись.

In [17]:
compressions = ["snappy", "gzip", "brotli"]
for compression in compressions:
    df.to_parquet("primer_" + compression + ".parquet", compression=compression, index=False)
print("Файлы сохранены")

Файлы сохранены


## 4.2

Сравните размеры полученных файлов с CSV

In [18]:
print("CSV:", os.path.getsize("primer.csv"), "байт")
for compression in compressions:
    print(compression + ":", os.path.getsize("primer_" + compression + ".parquet"), "байт")

CSV: 421984 байт
snappy: 160915 байт
gzip: 120271 байт
brotli: 112117 байт


## 4.3

Сравните время чтения Parquet и CSV

In [19]:
start = perf_counter()
for i in range(5):
    pd.read_csv("primer.csv", parse_dates=["hiredate"])
print("CSV:", round((perf_counter() - start) / 5, 6), "с")

for compression in compressions:
    start = perf_counter()
    for i in range(5):
        pd.read_parquet("primer_" + compression + ".parquet")
    print(compression + ":", round((perf_counter() - start) / 5, 6), "с")

CSV: 0.002685 с
snappy: 0.049638 с
gzip: 0.003967 с
brotli: 0.003399 с


## 4.4

Продемонстрируйте выборочное чтение столбцов (column pruning) - уникальная возможность Parquet

In [20]:
print(pd.read_parquet("primer_snappy.parquet", columns=["name", "salary"]).head())

    name  salary
0  Pavel  109841
1  Pavel   68010
2   Anna   77485
3  Maria  127866
4  Maria  144693


## 4.5

Проверьте сохранение типов данных

In [21]:
for compression in compressions:
    df_parquet = pd.read_parquet("primer_" + compression + ".parquet")
    print(compression, "- типы совпадают:", df.dtypes.equals(df_parquet.dtypes))
    print("Данные совпадают:", df.equals(df_parquet))
print(df_parquet.dtypes)

snappy - типы совпадают: True
Данные совпадают: True
gzip - типы совпадают: True
Данные совпадают: True
brotli - типы совпадают: True
Данные совпадают: True
id                     int64
name                     str
age                    int32
salary                 int32
department               str
hiredate      datetime64[us]
is_active               bool
dtype: object


## 5.1

Сохраните два DataFrame (например, df и агрегированный) в один HDF5-файл под разными ключами

In [29]:
df_group = df.groupby("department")["salary"].mean().reset_index()

df.to_hdf("primer.h5", key="employees", mode="w", format="table", data_columns=["salary"])
df_group.to_hdf("primer.h5", key="departments", mode="a")

with pd.HDFStore("primer.h5") as store:
    print(store.keys())

['/departments', '/employees']


## 5.2

Прочитайте только один из них по ключу

In [30]:
print(pd.read_hdf("primer.h5", key="departments"))

  department        salary
0    Finance  95217.709233
1         HR  95279.788490
2         IT  94953.878530


## 5.3

Сравните форматы fixed и table (последний поддерживает запросы через where)

In [31]:
df.to_hdf("primer_fixed.h5", key="data", mode="w", format="fixed")
df.to_hdf("primer_table.h5", key="data", mode="w", format="table", data_columns=["salary"])

print("fixed:", os.path.getsize("primer_fixed.h5"), "байт")
print("table:", os.path.getsize("primer_table.h5"), "байт")
print("fixed - данные совпадают:", df.equals(pd.read_hdf("primer_fixed.h5", key="data")))
print("table - данные совпадают:", df.equals(pd.read_hdf("primer_table.h5", key="data")))

fixed: 2589496 байт
table: 577281 байт
fixed - данные совпадают: True
table - данные совпадают: True


fixed используется для сохранения и чтения всей таблицы. table поддерживает выборочное чтение, добавление строк и запросы через where.

## 5.4

Выполните запрос к таблице HDF5 без загрузки всего файла в память

In [32]:
result = pd.read_hdf("primer_table.h5", key="data", where="salary > 120000")
print(result.head())
print("Количество строк:", len(result))

    id   name  age  salary department   hiredate  is_active
3    4  Maria   60  127866    Finance 2023-12-30       True
4    5  Maria   55  144693    Finance 2023-01-10       True
7    8   Olga   24  125214         HR 2021-06-18      False
11  12  Pavel   60  136114         HR 2021-12-28       True
14  15  Maria   56  130229         IT 2021-06-18       True
Количество строк: 2746


## 5.5

Сравните размеры файлов HDF5 и CSV

In [33]:
print("CSV:", os.path.getsize("primer.csv"), "байт")
print("HDF5 fixed:", os.path.getsize("primer_fixed.h5"), "байт")
print("HDF5 table:", os.path.getsize("primer_table.h5"), "байт")
print("HDF5 с двумя таблицами:", os.path.getsize("primer.h5"), "байт")

CSV: 421984 байт
HDF5 fixed: 2589496 байт
HDF5 table: 577281 байт
HDF5 с двумя таблицами: 1640185 байт


## 6.1

Программно соберите метрики: размер файла, время записи, время чтения для каждого формата

Для Parquet требуется `pyarrow`, для HDF5 - `tables`. Если библиотека не установлена, для соответствующего формата измерения не выполняются.

In [36]:
formats = ["CSV", "JSON", "Parquet", "HDF5"]
files = ["test.csv", "test.json", "test.parquet", "test.h5"]
metrics = []

for fmt, filename in zip(formats, files):
    start = perf_counter()

    for i in range(5):
        if fmt == "CSV":
            df.to_csv(filename, index=False)
        elif fmt == "JSON":
            df.to_json(filename, orient="records", date_format="iso")
        elif fmt == "Parquet":
            df.to_parquet(filename, compression="snappy", index=False)
        else:
            df.to_hdf(filename, key="data", mode="w", format="table")

    write_time = (perf_counter() - start) / 5

    start = perf_counter()

    for i in range(5):
        if fmt == "CSV":
            df_read = pd.read_csv(filename, parse_dates=["hiredate"])
            df_read = df_read.astype(df.dtypes.to_dict())
        elif fmt == "JSON":
            df_read = pd.read_json(filename, orient="records", dtype=False)
            df_read["hiredate"] = pd.to_datetime(df_read["hiredate"])
            df_read = df_read.astype(df.dtypes.to_dict())
        elif fmt == "Parquet":
            df_read = pd.read_parquet(filename)
        else:
            df_read = pd.read_hdf(filename, key="data")

    read_time = (perf_counter() - start) / 5

    metrics.append([fmt, os.path.getsize(filename), write_time, read_time])

    print(fmt, "- данные совпадают:", df.equals(df_read))

CSV - данные совпадают: True
JSON - данные совпадают: True
Parquet - данные совпадают: True
HDF5 - данные совпадают: True


## 6.2

Постройте итоговый DataFrame-отчёт

Для сравнения используется один и тот же DataFrame. Время записи и чтения усредняется по 5 запускам. Для Parquet используется snappy, для HDF5 - table.

In [37]:
report = pd.DataFrame(metrics, columns=["Формат", "Размер (байт)", "Запись (с)", "Чтение (с)"])
print(report.to_string(index=False, na_rep="Не измерено", formatters={
    "Размер (байт)": "{:.0f}".format,
    "Запись (с)": "{:.6f}".format,
    "Чтение (с)": "{:.6f}".format
}))

 Формат Размер (байт) Запись (с) Чтение (с)
    CSV        421984   0.009281   0.004813
   JSON       1251935   0.009088   0.028265
Parquet        160915   0.005903   0.003663
   HDF5        548778   0.015378   0.010213


## 6.3

Ответьте на вопросы: какой формат выбрать для аналитики? Для обмена данными? Для научных расчётов?

Для аналитики - Parquet, так как он хранит данные по столбцам, поддерживает сжатие и выборочное чтение.

Для обмена данными - CSV, так как его можно открыть в разных программах. Для вложенных данных и API лучше использовать JSON.

Для научных расчётов - HDF5, так как он позволяет хранить несколько объектов в одном файле и работать с большими наборами данных.

Размер файлов и скорость работы зависят от набора данных и параметров сохранения.